# Análise CPGF — Cartão de Pagamento do Governo Federal

**TP Banco de Dados — DCC/UFMG**  
Grupo: Lucas Soares Benfica · Vinícius Cardoso Antunes · Pedro Soares Pinto

**Dataset:** CPGF (Portal da Transparência — https://portaldatransparencia.gov.br/download-de-dados/cpgf)

Este notebook reproduz o pipeline completo:
1. Carga e limpeza do CSV bruto
2. Validação do modelo relacional (PostgreSQL ou SQLite)
3. Execução das 5 consultas analíticas (Etapa 4)
4. Geração das visualizações e discussão crítica

> Para reproduzir do zero, execute as células em ordem. Pré-requisitos no `README.md`.

## 1. Setup e configurações

In [ ]:
import sys, os, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Inclui /scripts no path para reusar funções da carga
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))

DB_PATH = Path('../data/processed/cpgf.db')  # ajuste se necessário
CSV_PATH = Path('../data/raw/AMOSTRA_CPGF.csv')  # ou o CSV real do Portal

pd.options.display.float_format = '{:,.2f}'.format

## 2. Carga (apenas se o .db ainda não existir)

In [ ]:
if not DB_PATH.exists():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    cmd = f"python ../scripts/carregar_dados.py --csv {CSV_PATH} --sqlite {DB_PATH}"
    print('Executando:', cmd)
    os.system(cmd)
else:
    print(f'Base ja existe em {DB_PATH}')

conn = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT COUNT(*) AS transacoes FROM transacao", conn)

## 3. Sanity check — visão geral

In [ ]:
Q0 = '''
SELECT
    COUNT(*)                          AS total_transacoes,
    COUNT(DISTINCT cpf_portador)      AS portadores_unicos,
    COUNT(DISTINCT cpf_cnpj_favorecido) AS favorecidos_unicos,
    COUNT(DISTINCT codigo_ug)         AS ugs_unicas,
    MIN(valor)                        AS valor_min,
    MAX(valor)                        AS valor_max,
    ROUND(AVG(valor), 2)              AS valor_medio,
    ROUND(SUM(valor), 2)              AS valor_total
FROM transacao
'''
pd.read_sql(Q0, conn)

## P1 — Quais ministérios mais gastam via CPGF?
*Cobre: JOIN (3 tabelas) + GROUP BY + agregações*

In [ ]:
Q1 = '''
SELECT
    os.nome                         AS ministerio,
    COUNT(t.id)                     AS qtd_transacoes,
    ROUND(SUM(t.valor), 2)          AS gasto_total_brl,
    ROUND(AVG(t.valor), 2)          AS ticket_medio_brl,
    COUNT(DISTINCT t.cpf_portador)  AS portadores_distintos
FROM transacao         t
JOIN unidade_gestora   ug   ON ug.codigo   = t.codigo_ug
JOIN orgao_subordinado osub ON osub.codigo = ug.codigo_orgao_subordinado
JOIN orgao_superior    os   ON os.codigo   = osub.codigo_orgao_superior
WHERE t.valor > 0
GROUP BY os.nome
ORDER BY gasto_total_brl DESC
'''
df1 = pd.read_sql(Q1, conn)
df1

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
df1.sort_values('gasto_total_brl').plot.barh(
    x='ministerio', y='gasto_total_brl', ax=ax, color='#1f4e79', legend=False)
ax.set_xlabel('Gasto total (R$)')
ax.set_title('P1 — Gasto total via CPGF por Ministério')
plt.tight_layout()

## P2 — Sazonalidade dos gastos
*Cobre: GROUP BY composto + agregação temporal*

In [ ]:
Q2 = '''
SELECT ano_extrato, mes_extrato,
       COUNT(*) AS qtd_transacoes,
       ROUND(SUM(valor), 2) AS gasto_total
FROM transacao WHERE valor > 0
GROUP BY ano_extrato, mes_extrato
ORDER BY ano_extrato, mes_extrato
'''
df2 = pd.read_sql(Q2, conn)
df2['periodo'] = df2['ano_extrato'].astype(str) + '-' + df2['mes_extrato'].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df2['periodo'], df2['gasto_total'], marker='o', color='#c0504d', linewidth=2)
ax.tick_params(axis='x', rotation=45)
ax.set_title('P2 — Evolução temporal do gasto CPGF')
ax.set_ylabel('Gasto total (R$)')
ax.grid(alpha=0.3)
plt.tight_layout()

## P3 — Top-10 favorecidos
*Cobre: JOIN + GROUP BY + HAVING + LIMIT*

In [ ]:
Q3 = '''
SELECT f.nome AS favorecido, f.tipo,
       COUNT(t.id) AS qtd_pagamentos,
       ROUND(SUM(t.valor), 2) AS total_recebido,
       ROUND(AVG(t.valor), 2) AS ticket_medio
FROM transacao  t
JOIN favorecido f ON f.cpf_cnpj = t.cpf_cnpj_favorecido
WHERE t.valor > 0
GROUP BY f.nome, f.tipo
HAVING COUNT(t.id) >= 5
ORDER BY total_recebido DESC
LIMIT 10
'''
df3 = pd.read_sql(Q3, conn)
df3

## P4 — Distribuição por tipo de transação
*Cobre: JOIN + MIN/MAX/AVG/SUM/COUNT*

In [ ]:
Q4 = '''
SELECT tt.descricao AS tipo_transacao,
       COUNT(t.id) AS qtd,
       ROUND(SUM(t.valor), 2) AS total,
       ROUND(MIN(t.valor), 2) AS minimo,
       ROUND(MAX(t.valor), 2) AS maximo,
       ROUND(AVG(t.valor), 2) AS media
FROM transacao t
JOIN tipo_transacao tt ON tt.codigo = t.codigo_tipo_transacao
GROUP BY tt.descricao
ORDER BY total DESC
'''
pd.read_sql(Q4, conn)

## P5 — Top-3 portadores por ministério (consulta avançada)
*Cobre: CTE + WINDOW FUNCTION (RANK OVER PARTITION BY)*

In [ ]:
Q5 = '''
WITH gasto_por_portador AS (
    SELECT t.cpf_portador, p.nome AS nome_portador, os.nome AS ministerio,
           SUM(t.valor) AS gasto_portador
    FROM transacao         t
    JOIN portador          p    ON p.cpf = t.cpf_portador
    JOIN unidade_gestora   ug   ON ug.codigo = t.codigo_ug
    JOIN orgao_subordinado osub ON osub.codigo = ug.codigo_orgao_subordinado
    JOIN orgao_superior    os   ON os.codigo = osub.codigo_orgao_superior
    WHERE t.valor > 0
    GROUP BY t.cpf_portador, p.nome, os.nome
),
estatisticas AS (
    SELECT cpf_portador, nome_portador, ministerio, gasto_portador,
           AVG(gasto_portador)  OVER (PARTITION BY ministerio) AS media_min,
           RANK()               OVER (PARTITION BY ministerio
                                       ORDER BY gasto_portador DESC) AS rank_no_min
    FROM gasto_por_portador
)
SELECT ministerio, nome_portador, ROUND(gasto_portador, 2) AS gasto_brl,
       ROUND(media_min, 2) AS media_min,
       ROUND(gasto_portador / media_min, 2) AS razao_vs_media,
       rank_no_min
FROM estatisticas WHERE rank_no_min <= 3
ORDER BY ministerio, rank_no_min
'''
pd.read_sql(Q5, conn)

## Discussão crítica

Ver `docs/04_eda_discussao.md` para a discussão completa dos achados, padrões, anomalias e limitações da análise.

In [ ]:
conn.close()